# Day 12 — Imbalanced Data & Advanced Metrics

**Agri-AI Bootcamp** · Phase 3 of 5: *CNNs & Crop Disease* · Days 9–12

> Drop 90% of diseased images, watch metrics fail, then fix with CrossEntropyLoss(weight=...).

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

When one class dominates, a model can score high accuracy by always predicting it. Weighted loss makes mistakes on the rare class cost more. Precision-Recall curves judge the rare class without the majority class inflating the score.

### 🌾 Agriculture example

If only 2% of leaves show a quarantine disease, a model that always says “healthy” is 98% accurate and useless. Weight the rare class higher and watch its recall.

> **Watch out:** Weights are usually inverse to class frequency; check their order matches the class indices.

## 2. Setup

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (confusion_matrix, classification_report,
                             precision_recall_curve, average_precision_score)
dev = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0)

## 3. Data

In [ ]:
# ---- Leaf image helper -------------------------------------------------------
# Real course dataset: PlantVillage / Tomato leaf images on Kaggle (needs a
# kaggle.json API token).  So that this notebook runs anywhere, we fall back to
# a small SYNTHETIC leaf set: green patches = "healthy", green patches with
# brown lesions = "diseased".  Swap in the real folder when you have it.
import numpy as np, torch

def synth_leaves(n=400, size=64, seed=0):
    rng = np.random.default_rng(seed)
    X = np.zeros((n, 3, size, size), dtype=np.float32)
    y = np.zeros(n, dtype=np.int64)
    for i in range(n):
        leaf = np.stack([rng.uniform(.10, .25, (size, size)),
                         rng.uniform(.45, .70, (size, size)),
                         rng.uniform(.10, .25, (size, size))])
        diseased = i % 2
        if diseased:                      # add 3-6 brown lesions
            for _ in range(rng.integers(3, 7)):
                cx, cy = rng.integers(8, size - 8, 2)
                r = rng.integers(4, 9)
                yy, xx = np.ogrid[:size, :size]
                m = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
                leaf[0][m], leaf[1][m], leaf[2][m] = .45, .28, .10
        X[i] = leaf
        y[i] = diseased
    return torch.tensor(X), torch.tensor(y)

CLASSES = ["healthy", "diseased"]
X_img, y_img = synth_leaves()
print("images:", tuple(X_img.shape), "| labels:", torch.bincount(y_img).tolist())

# ---- Make it realistically imbalanced: keep only 10% of the diseased leaves --
healthy  = (y_img == 0).nonzero().flatten()
diseased = (y_img == 1).nonzero().flatten()
keep = torch.cat([healthy, diseased[:max(1, len(diseased)//10)]])
keep = keep[torch.randperm(len(keep))]
Xi, yi = X_img[keep], y_img[keep]
cut = int(.75*len(yi))
tr_dl = DataLoader(TensorDataset(Xi[:cut], yi[:cut]), batch_size=32, shuffle=True)
te_dl = DataLoader(TensorDataset(Xi[cut:], yi[cut:]), batch_size=64)
cnt = torch.bincount(yi[:cut], minlength=2)
print(f"train counts  healthy {cnt[0].item()} | diseased {cnt[1].item()} "
      f"({100*cnt[1].item()/cnt.sum().item():.1f}% rare class)")
print(f"a model that ALWAYS says 'healthy' scores "
      f"{(yi[cut:]==0).float().mean().item():.3f} accuracy and is useless")

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
cnt=torch.bincount(ytr)
w=cnt.sum()/(len(cnt)*cnt)          # rare class gets a bigger weight
loss_fn=nn.CrossEntropyLoss(weight=w.float().to(dev))
```

## 5. Worked solution

In [ ]:
def make_cnn():
    torch.manual_seed(0)
    return nn.Sequential(nn.Conv2d(3,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 2)).to(dev)

def run(loss_fn, epochs=12, tag=""):
    net = make_cnn(); opt = torch.optim.Adam(net.parameters(), 1e-3)
    for _ in range(epochs):
        net.train()
        for xb, yb in tr_dl:
            opt.zero_grad(); loss_fn(net(xb.to(dev)), yb.to(dev)).backward(); opt.step()
    net.eval(); probs, true = [], []
    with torch.no_grad():
        for xb, yb in te_dl:
            probs.append(torch.softmax(net(xb.to(dev)), 1)[:, 1].cpu()); true.append(yb)
    p = torch.cat(probs).numpy(); t = torch.cat(true).numpy()
    pred = (p >= .5).astype(int)
    tp = ((pred==1)&(t==1)).sum(); fn = ((pred==0)&(t==1)).sum(); fp = ((pred==1)&(t==0)).sum()
    rec = tp/max(tp+fn,1); prec = tp/max(tp+fp,1)
    print(f"{tag:<22} acc {(pred==t).mean():.3f} | rare-class recall {rec:.3f} "
          f"precision {prec:.3f} | AP {average_precision_score(t, p):.3f}")
    print("   confusion [[TN FP][FN TP]]:", confusion_matrix(t, pred).tolist())
    return p, t

# 1) plain cross-entropy -- high accuracy, misses the disease
p0, t0 = run(nn.CrossEntropyLoss(), tag="plain CE")

# 2) weighted cross-entropy: weight INVERSE to class frequency
w = cnt.sum() / (len(cnt) * cnt.float())          # order matches class indices 0,1
print(f"\nclass weights (index order {list(range(len(w)))}): {w.numpy().round(3)}")
p1, t1 = run(nn.CrossEntropyLoss(weight=w.to(dev)), tag="weighted CE")

# 3) oversample the rare class in the DataLoader instead
from torch.utils.data import WeightedRandomSampler
sample_w = w[yi[:cut]].double()
sampler  = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)
tr_dl = DataLoader(TensorDataset(Xi[:cut], yi[:cut]), batch_size=32, sampler=sampler)
p2, t2 = run(nn.CrossEntropyLoss(), tag="oversampling")

# ---- Precision-Recall beats ROC when the positive class is rare -------------
plt.figure(figsize=(6, 4))
for name, (p, t) in [("plain CE", (p0, t0)), ("weighted CE", (p1, t1)), ("oversample", (p2, t2))]:
    pr, rc, _ = precision_recall_curve(t, p)
    plt.plot(rc, pr, label=f"{name} (AP={average_precision_score(t, p):.2f})")
plt.xlabel("recall (diseased leaves found)"); plt.ylabel("precision")
plt.title("PR curve -- the majority class cannot inflate this"); plt.legend(); plt.show()

print("\nFor a quarantine disease, recall is the number the agronomist cares about:")
print("a missed outbreak costs far more than one unnecessary inspection.")

## 6. Your turn

1. Set the weights the WRONG way round (frequent class heavier) and look at the recall. Order matters.
2. Pick the threshold from the PR curve that gives recall >= 0.9, and report the precision you pay for it.
3. Try `torchvision.ops.sigmoid_focal_loss` as a third option.

In [ ]:
# TODO 1: reversed weights -- the bug you are learning to spot
w_bad = (cnt.float() / cnt.sum())
_ = run(nn.CrossEntropyLoss(weight=w_bad.to(dev)), tag="WRONG weights")

# TODO 2: threshold for recall >= 0.9
pr, rc, th = precision_recall_curve(t1, p1)
ok = rc[:-1] >= 0.90
if ok.any():
    i = np.argmax(pr[:-1][ok]); print("threshold", th[ok][i].round(3), "precision", pr[:-1][ok][i].round(3))

# TODO 3: focal loss

## 7. Dataset and references

**Dataset:** [Kaggle: PlantVillage leaf images](https://www.kaggle.com/datasets/emmarex/plantdisease)

**References**

- [Google ML Crash Course: imbalanced data](https://developers.google.com/machine-learning/crash-course/overfitting/imbalanced-datasets)

**Course material:** Google ML Crash Course: Imbalanced Data

---
**Next:** Day 13 — Bounding Box Foundations